* > **This implementation is based on the framework proposed in the paper "DIAA: Distillation with Illumination-aware Adaptive Attention for Low-Light Image Enhancement," leveraging a distillation-driven approach combined with illumination-aware frequency processing to enhance visibility and suppress complex noise. The training and testing datasets utilize paired low-light and normal-light evaluation images from the LOL dataset. Paper details can be found [here](https://www.ijci.zu.edu.eg/index.php/ijci/article/download/146/112/315).**
* > **Training dataset consists of 485 pairs of low-light and normal-light images.**
* > **Testing dataset consists of 15 pairs of low-light and normal-light images used to evaluate the model's enhancement and noise-reduction performance.**

# Importing required libraries

In [ ]:
import argparse
import re
import os, time, datetime
import numpy as np
from skimage import img_as_ubyte
from keras.models import load_model, model_from_json
from skimage.metrics import peak_signal_noise_ratio,structural_similarity
from skimage.io import imread, imsave
import tensorflow as tf
import matplotlib.pyplot as plt
from skimage import img_as_ubyte
import os, glob, datetime
from keras.layers import  Input,Conv2D,BatchNormalization,Activation,Subtract
from keras.models import Model, load_model
from keras.callbacks import CSVLogger, ModelCheckpoint, LearningRateScheduler
from keras.optimizers import Adam
import keras.backend as K
from skimage.metrics import structural_similarity as compare_ssim
from tensorflow.keras import layers
from tensorflow.keras.losses import MeanSquaredError, KLDivergence
from tensorflow.keras.applications import VGG16, ResNet50, InceptionV3
from tensorflow.keras.layers import Add
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import cv2
from skimage.util import img_as_float,random_noise
from PIL import Image
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import PReLU, Conv2DTranspose, Dropout, concatenate
from tensorflow.keras.layers import MaxPooling2D, Dropout, UpSampling2D, concatenate, LeakyReLU
from tensorflow.keras.regularizers import l1_l2
from tensorflow.keras.metrics import MeanIoU
from scipy.ndimage import convolve
from skimage.metrics import mean_squared_error
import pandas as pd
from skimage import color
from scipy.special import gamma

# Preprocessing Training data

In [ ]:
import glob
import os
import cv2
import numpy as np
import datetime
import random
from keras.callbacks import ModelCheckpoint, CSVLogger, LearningRateScheduler, ReduceLROnPlateau
from keras.models import load_model
import tensorflow as tf
import gc

patch_size, stride = 64, 64
aug_times = 1
scales = [1, 0.9, 0.8, 0.7]

batch_size = 8
low_light_dir = '../input/lol-dataset/lol_dataset/our485/low'
high_light_dir ='../input/lol-dataset/lol_dataset/our485/high'

def log(*args, **kwargs):
    print(datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S:"), *args, **kwargs)

def lr_schedule(epoch):
    initial_lr = args.lr
    if epoch <= 30:
        lr = initial_lr
    elif epoch <= 60:
        lr = initial_lr / 10
    elif epoch <= 80:
        lr = initial_lr / 20
    else:
        lr = initial_lr / 20
    log('current learning rate is %2.8f' % lr)
    return lr

def data_aug(img, mode=0):
    if mode == 0:
        return img
    elif mode == 1:
        return np.flipud(img)
    elif mode == 2:
        return np.rot90(img)
    elif mode == 3:
        return np.flipud(np.rot90(img))
    elif mode == 4:
        return np.rot90(img, k=2)
    elif mode == 5:
        return np.flipud(np.rot90(img, k=2))
    elif mode == 6:
        return np.rot90(img, k=3)
    elif mode == 7:
        return np.flipud(np.rot90(img, k=3))

def gen_patches_pair(low_img_path, high_img_path):
    # Read and resize images
    img_low = cv2.imread(low_img_path)
    img_high = cv2.imread(high_img_path)
    # img_low = cv2.resize(img_low, (256, 256), interpolation=cv2.INTER_CUBIC)
    # img_high = cv2.resize(img_high, (256, 256), interpolation=cv2.INTER_CUBIC)
    
    patches_low, patches_high = [], []
    h, w, _ = img_low.shape
    
    for s in scales:
        h_scaled, w_scaled = int(h * s), int(w * s)
        low_scaled = cv2.resize(img_low, (w_scaled, h_scaled), interpolation=cv2.INTER_CUBIC)
        high_scaled = cv2.resize(img_high, (w_scaled, h_scaled), interpolation=cv2.INTER_CUBIC)
        
        for i in range(0, h_scaled - patch_size + 1, stride):
            for j in range(0, w_scaled - patch_size + 1, stride):
                patch_low = low_scaled[i:i+patch_size, j:j+patch_size]
                patch_high = high_scaled[i:i+patch_size, j:j+patch_size]
                for _ in range(aug_times):
                    mode = random.randint(0, 7)
                    patches_low.append(data_aug(patch_low, mode))
                    patches_high.append(data_aug(patch_high, mode))
    return patches_low, patches_high

def datagenerator(low_light_dir, high_light_dir, verbose=False):
    # List all files
    low_files = sorted(glob.glob(os.path.join(low_light_dir, '*.png')))
    high_files = sorted(glob.glob(os.path.join(high_light_dir, '*.png')))
    
    assert len(low_files) == len(high_files), "Mismatch between low and high-light images."
    
    data_low, data_high = [], []
    for i in range(len(low_files)):
        low_patches, high_patches = gen_patches_pair(low_files[i], high_files[i])
        data_low.append(low_patches)
        data_high.append(high_patches)
        if verbose:
            print(f'Processed {i+1}/{len(low_files)} image pairs.')

    data_low = np.array(data_low, dtype='uint8').reshape(-1, patch_size, patch_size, 3)
    data_high = np.array(data_high, dtype='uint8').reshape(-1, patch_size, patch_size, 3)
    
    # Discard last incomplete batch if exists
    discard_n = len(data_low) - (len(data_low) // batch_size) * batch_size
    if discard_n > 0:
        data_low = data_low[:-discard_n]
        data_high = data_high[:-discard_n]
    
    log('Data generation finished, total patches:', len(data_low))
    return data_low, data_high

def train_datagen(epoch_iter=2000, epoch_num=5, batch_size=128, low_light_dir=None, high_light_dir=None):
    n_count = 0
    while True:
        if n_count == 0:
            xs_low, xs_high = datagenerator(low_light_dir, high_light_dir)
            # assert len(xs_low) % batch_size == 0, "Last batch must have full batch size."
            xs_low = xs_low.astype('float32') / 255.0
            xs_high = xs_high.astype('float32') / 255.0
            indices = list(range(xs_low.shape[0]))
            n_count = 1
        
        for _ in range(epoch_num):
            random.shuffle(indices)
            for i in range(0, len(indices), batch_size):
                batch_x = xs_low[indices[i:i+batch_size]]
                batch_y = xs_high[indices[i:i+batch_size]]
                yield batch_x, batch_y

# -- MODEL TRAINING SETUP --

# Training Phase

In [ ]:
## Params
parser = argparse.ArgumentParser()
parser.add_argument('--model', default='DIAA', type=str, help='choose a type of model')
parser.add_argument('--batch_size', default=128, type=int, help='batch size')
parser.add_argument('--sigma', default=25, type=int, help='noise level')
parser.add_argument('--epoch', default=10, type=int, help='number of train epoches')
parser.add_argument('--lr', default=1e-3, type=float, help='initial learning rate for Adam')
parser.add_argument('--save_every', default=500, type=int, help='save model at every x epoches')
args = parser.parse_args(args=[])

save_dir = os.path.join('./','models',args.model+'_'+'sigma'+str(args.sigma)) 

if not os.path.exists(save_dir):
  os.makedirs(save_dir)

# DIAA model

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from keras.utils import register_keras_serializable

@register_keras_serializable()
class ClipConstraint(tf.keras.constraints.Constraint):
    def __init__(self, min_value, max_value):
        self.min_value = min_value
        self.max_value = max_value

    def __call__(self, w):
        return tf.clip_by_value(w, self.min_value, self.max_value)

    def get_config(self):
        return {'min_value': self.min_value, 'max_value': self.max_value}

@register_keras_serializable()
def swish_activation(x):
    return x * tf.keras.backend.sigmoid(x)

@register_keras_serializable()
class GammaCorrectionLayer(layers.Layer):
    def __init__(self, initial_gamma=0.4, **kwargs):
        super().__init__(**kwargs)
        self.initial_gamma = initial_gamma

    def build(self, input_shape):
        self.gamma = self.add_weight(
            name='gamma',
            shape=(),
            initializer=tf.constant_initializer(self.initial_gamma),
            trainable=True,
            constraint=ClipConstraint(0.1, 2.0)
        )

    def call(self, inputs):
        inputs = tf.clip_by_value(inputs, 0.0, 1.0)
        return tf.pow(inputs, self.gamma)

@register_keras_serializable()
class AdaptiveExposureCorrection(layers.Layer):
    def build(self, input_shape):
        self.exposure_gain = self.add_weight(
            name='exposure_gain',
            shape=(1,),
            initializer='zeros',
            trainable=True
        )

    def call(self, inputs):
        return inputs * tf.exp(self.exposure_gain)

@register_keras_serializable()
def squeeze_excite_block(x, ratio=8):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Reshape((1, 1, filters))(se)
    se = layers.Dense(filters // ratio, activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    return x * se

@register_keras_serializable()
class SpatialAttention(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.concat = layers.Concatenate()
        self.conv = layers.Conv2D(1, 5, padding='same', activation='sigmoid')
        self.multiply = layers.Multiply()

    def build(self, input_shape):
        super().build(input_shape)

    def call(self, x):
        avg_out = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_out = tf.reduce_max(x, axis=-1, keepdims=True)
        concat = self.concat([avg_out, max_out])
        attention = self.conv(concat)
        return self.multiply([x, attention])

@register_keras_serializable()
def illumination_estimation(inputs):
    x1 = layers.Conv2D(32, 3, strides=1, padding='same')(inputs)
    x2 = layers.Conv2D(32, 3, strides=2, padding='same')(inputs)
    x3 = layers.Conv2D(32, 3, strides=4, padding='same')(inputs)
    x2 = layers.Conv2D(32, 3, padding='same')(x2)
    x3 = layers.Conv2D(32, 3, padding='same')(x3)
    x2 = layers.UpSampling2D(2)(x2)
    x3 = layers.UpSampling2D(4)(x3)
    x = layers.Concatenate()([x1, x2, x3])
    illum = layers.Conv2D(1, 3, padding='same', activation='sigmoid')(x)
    return illum

@register_keras_serializable()
def noise_aware_residual_block(x):
    identity = x
    channels = x.shape[-1]
    noise = layers.Conv2D(channels//2, 3, padding='same')(x)
    noise = layers.Conv2D(channels, 3, padding='same')(noise)
    x = layers.Conv2D(channels, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(swish_activation)(x)
    x = layers.subtract([x, noise])
    x = squeeze_excite_block(x)
    x = SpatialAttention()(x)
    return layers.Add()([identity, x])

@register_keras_serializable()
def multi_scale_block(x):
    d1 = layers.Conv2D(32, 3, dilation_rate=1, padding='same')(x)
    d2 = layers.Conv2D(32, 3, dilation_rate=2, padding='same')(x)
    x = layers.Concatenate()([d1, d2])
    return layers.Conv2D(64, 1, padding='same')(x)

@register_keras_serializable()
class FrequencyAwareBlock(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.global_avg = layers.GlobalAvgPool2D(keepdims=True)
        self.conv_low = layers.Conv2D(64, 1, activation=swish_activation)
        self.upsample = layers.UpSampling2D(interpolation='bilinear')
        self.conv_high = layers.Conv2D(64, 3, padding='same')
        self.concat = layers.Concatenate()
        self.conv_fuse = layers.Conv2D(64, 1, padding='same')

    def call(self, x):
        low = self.global_avg(x)
        low = self.conv_low(low)
        low = self.upsample(low)
        low = tf.image.resize(low, [tf.shape(x)[1], tf.shape(x)[2]], method='bilinear')
        high = self.conv_high(x)
        combined = self.concat([low, high])
        return self.conv_fuse(combined)

@register_keras_serializable()
def transformer_cnn_branch(x):
    cnn = layers.Conv2D(64, 3, padding='same', activation=swish_activation)(x)
    attn = layers.Conv2D(64, 3, padding='same', activation='relu')(cnn)
    attn = SpatialAttention()(attn)
    return layers.Concatenate()([cnn, attn])

def DIAA():
    inputs = layers.Input(shape=(None, None, 3))
    illum_map = illumination_estimation(inputs)
    x = GammaCorrectionLayer(initial_gamma=0.4)(inputs)
    x = AdaptiveExposureCorrection()(x)
    x_low = layers.AveragePooling2D(2)(x)
    x_high = x - layers.UpSampling2D(2)(x_low)
    x_low = layers.concatenate([x_low, layers.AveragePooling2D(2)(illum_map)])
    x_low = layers.Conv2D(32, 3, padding='same')(x_low)
    x_high = layers.Conv2D(32, 3, padding='same')(x_high)
    x_low = layers.UpSampling2D(2)(x_low)
    x = layers.Concatenate()([x_low, x_high])
    x = layers.Conv2D(64, 1, padding='same')(x)
    cnn_branch = multi_scale_block(x)
    transformer_branch = transformer_cnn_branch(x)
    x = layers.Concatenate()([cnn_branch, transformer_branch])
    for _ in range(3):
        x = noise_aware_residual_block(x)
    x = FrequencyAwareBlock()(x)
    x = layers.Conv2D(32, 3, padding='same')(x)
    x = layers.Activation(swish_activation)(x)
    outputs = layers.Conv2D(3, 3, padding='same', activation='sigmoid')(x)
    return Model(inputs, outputs)


In [ ]:
# Teacher model
def build_teacher_model(input_shape=(None, None, 3)):
    # Load the pre-trained VGG16 model without the top classification layer
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)

    # Freeze the layers in the base model
    base_model.trainable = False

    # Extract features up to a certain layer (e.g., the last convolutional layer)
    intermediate_layer = base_model.get_layer('conv5_block3_out')
    teacher_outputs = intermediate_layer.output
    teacher_model = tf.keras.Model(inputs=base_model.input, outputs=teacher_outputs)

    return teacher_model

In [ ]:
teacher_model = build_teacher_model()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999)
teacher_model.compile(optimizer=optimizer, loss='mean_squared_error', metrics=['accuracy'])

# Load model from last Checkpoint

In [ ]:
def findLastCheckpoint(save_dir):
    file_list = glob.glob(os.path.join(save_dir,'model_*.keras'))  # get name list of all .hdf5 files
    if file_list:
        epochs_exist = []
        for file_ in file_list:
            result = re.findall(".*model_(.*).keras.*",file_)
            epochs_exist.append(int(result[0]))
        initial_epoch=max(epochs_exist)   
    else:
        initial_epoch = 0
    return initial_epoch

# Learning rate, log file, train data generation

# Loss Function

In [ ]:
@register_keras_serializable()
def peak_signal_noise_ratio(y_true, y_pred):
    return tf.image.psnr(y_pred, y_true, max_val=1.0)

In [ ]:
from keras.saving import register_keras_serializable
from tensorflow.keras.applications import VGG16, VGG19, ResNet50, InceptionV3

# Custom loss function for knowledge distillation
def custom_loss(distillation_weight=0.1, temperature=1.0, epsilon=1e-10):

    # Pre-trained VGG model for Perceptual Loss
    vgg = VGG19(include_top=False, weights='imagenet', input_shape=(patch_size, patch_size, 3))
    vgg.trainable = False

    @register_keras_serializable()
    def perceptual_loss(y_true, y_pred):
        # Extract feature maps from certain layers of VGG
        vgg_true = vgg(y_true)
        vgg_pred = vgg(y_pred)
        return tf.reduce_mean(tf.square(vgg_true - vgg_pred))

    @register_keras_serializable()
    def illumination_smoothness_loss(y_pred):
        # Compute the gradients (x and y directions)
        gradient_x = tf.abs(y_pred[:, :-1, :, :] - y_pred[:, 1:, :, :])
        gradient_y = tf.abs(y_pred[:, :, :-1, :] - y_pred[:, :, 1:, :])
    
        # Add padding to gradients to make their shapes match
        gradient_x = tf.pad(gradient_x, [[0, 0], [0, 1], [0, 0], [0, 0]])  # Pad the second dimension
        gradient_y = tf.pad(gradient_y, [[0, 0], [0, 0], [0, 1], [0, 0]])  # Pad the third dimension
    
        # Return the mean of the sum of gradients
        return tf.reduce_mean(gradient_x + gradient_y)

    @register_keras_serializable()
    def color_constancy_loss(y_pred):
        # Ensure that the color channels are balanced
        r, g, b = y_pred[:, :, :, 0], y_pred[:, :, :, 1], y_pred[:, :, :, 2]
        r_mean, g_mean, b_mean = tf.reduce_mean(r), tf.reduce_mean(g), tf.reduce_mean(b)
        return tf.reduce_mean(tf.abs(r_mean - g_mean) + tf.abs(g_mean - b_mean) + tf.abs(b_mean - r_mean))
    
    @register_keras_serializable()
    def exposure_control_loss(y_pred, target_exposure=0.6):
        # Control the exposure to a target level (e.g., 0.6 for medium exposure)
        exposure = tf.reduce_mean(y_pred)
        return tf.square(exposure - target_exposure)
    
    @register_keras_serializable()
    def loss(y_true, y_pred):
        # Generator loss (e.g., Mean Squared Error)
        gen_loss = tf.reduce_mean(tf.square(y_true - y_pred))  # MSE as the base generator loss

        # SSIM Loss
        ssim_loss = 1.0 - tf.image.ssim(y_true, y_pred, max_val=1.0)

        # L1 Loss
        l1_loss = tf.reduce_mean(tf.abs(y_true - y_pred))

        # Perceptual Loss
        percept_loss = perceptual_loss(y_true, y_pred)

        # Illumination Smoothness Loss
        illum_loss = illumination_smoothness_loss(y_pred)

        # Color Constancy Loss
        color_loss = color_constancy_loss(y_pred)

        # Exposure Control Loss
        exposure_loss = exposure_control_loss(y_pred)

        # PSNR calculation (for normalization)
        mse = tf.reduce_mean(tf.square(y_true - y_pred))
        psnr = tf.cond(tf.equal(mse, 0),
                       lambda: tf.constant(0, dtype=tf.float32),
                       lambda: 10.0 * tf.math.log(1.0 / mse) / tf.math.log(10.0))

        if teacher_model is not None:  # Check if teacher model is defined
            teacher_predictions = teacher_model(y_true, training=False)
            student_predictions = teacher_model(y_pred, training=False)

            # Scale the logits of teacher predictions using temperature
            scaled_teacher_logits = teacher_predictions / temperature
            scaled_student_logits = student_predictions / temperature

            # Apply softmax to get scaled predictions
            scaled_teacher_predictions = tf.nn.softmax(scaled_teacher_logits, axis=-1)
            scaled_student_predictions = tf.nn.softmax(scaled_student_logits, axis=-1)

            # Avoid division by zero by adding a small epsilon
            distillation_loss = tf.reduce_sum(
                scaled_teacher_predictions * (temperature ** 2) * 
                tf.math.log((scaled_teacher_predictions + epsilon) / (scaled_student_predictions + epsilon)),
                axis=-1
            )
            distillation_loss = tf.reduce_mean(distillation_loss)

            total_loss = (
                distillation_weight * distillation_loss +  # Adjust distillation weight if using a teacher model
                ssim_loss * 1 +                            # Increase SSIM loss weight for better structure
                percept_loss * 1 +                         # Increase Perceptual loss weight (adjust if blurry)
                illum_loss * 1 +                           # Decrease Illumination smoothness weight to reduce blur
                color_loss * 1 +                           # Increase Color Constancy loss weight for better color balance
                exposure_loss * 1                          # Adjust exposure loss to manage brightness
            ) 
        else:
            total_loss = (
                ssim_loss * 1 +                            # Adjust SSIM for structure preservation
                percept_loss * 1 +                         # Increase Perceptual loss weight for more details
                illum_loss * 1 +                           # Reduce illumination smoothness to lessen blurring
                color_loss * 1 +                           # Strengthen color balance handling
                exposure_loss * 1                          # Tweak exposure control for better brightness adjustment
            ) 

        return total_loss

    return loss

# Model Summary

In [ ]:
# model selection
model = DIAA()
model.summary()

# Testing phase

In [ ]:
# Clear previous session & free memory before loading model
import keras.backend as K
K.clear_session()
gc.collect()

initial_epoch = findLastCheckpoint(save_dir=save_dir)  # your own checkpoint finder

if initial_epoch > 0:
    log(f'Resuming from epoch {initial_epoch}')
    model = load_model(
        os.path.join(save_dir, f'model_{initial_epoch:03d}.keras'),
        compile=False,
        custom_objects={'GammaCorrectionLayer': GammaCorrectionLayer}
    )
else:
    model = DIAA()  # your model constructor

# Callbacks
learning_rate_reduction = ReduceLROnPlateau(
    monitor='peak_signal_noise_ratio',    
    patience=3, 
    verbose=1, 
    factor=0.8, 
    min_lr=1e-10,
)

checkpointer = ModelCheckpoint(
    os.path.join(save_dir, 'model_{epoch:03d}.keras'), 
    verbose=1, save_weights_only=False, save_freq='epoch'
)
csv_logger = CSVLogger(os.path.join(save_dir, 'log.csv'), append=True, separator=',')
lr_scheduler = LearningRateScheduler(lr_schedule)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001, beta_1=0.7, beta_2=0.999),
    # loss=custom_loss,
    loss= custom_loss(distillation_weight=.3, temperature=0.8),
    metrics=['mse', 'accuracy', peak_signal_noise_ratio]
)

history = model.fit(
    train_datagen(
        batch_size=args.batch_size,
        low_light_dir=low_light_dir,
        high_light_dir=high_light_dir
    ),
    steps_per_epoch=2000,
    epochs=args.epoch,
    verbose=1,
    initial_epoch=initial_epoch,
    callbacks=[checkpointer, csv_logger, learning_rate_reduction]

)

In [ ]:
from scipy.ndimage import convolve
from imageio import imread, imsave
from skimage.metrics import structural_similarity as compare_ssim
import numpy as np
import os
import cv2
import time
import pandas as pd
import argparse
from keras.models import load_model, model_from_json
import keras

def contrast_to_noise_ratio(enhanced_img, ground_truth_img):
    cnr_values = []
    for i in range(3):  # For each channel (R, G, B)
        mean_signal = np.mean(ground_truth_img[..., i])
        mean_noise = np.mean(enhanced_img[..., i] - ground_truth_img[..., i])
        std_noise = np.std(enhanced_img[..., i] - ground_truth_img[..., i])
        if std_noise < 1e-6:
            cnr_values.append(0)
        else:
            cnr_values.append(np.abs(mean_signal - mean_noise) / std_noise)
    return np.mean(cnr_values)

def equivalent_number_of_looks(denoised_img):
    mean_val = np.mean(denoised_img)
    std_val = np.std(denoised_img)
    if std_val == 0:
        return float('inf')
    return (mean_val / std_val) ** 2

def edge_preservation_index(denoised_image, image):
    edge_image = np.gradient(image)
    edge_denoised_image = np.gradient(denoised_image)
    diff = np.abs(edge_image[0] - edge_denoised_image[0]) + np.abs(edge_image[1] - edge_denoised_image[1])
    return np.mean(diff)

def root_mean_square_error(enhanced_img, ground_truth_img):
    return np.sqrt(np.mean((ground_truth_img - enhanced_img) ** 2))

def figure_of_merit(denoised_img, ground_truth_img, alpha=1):
    gradient_denoised = np.gradient(denoised_img)
    gradient_ground_truth = np.gradient(ground_truth_img)
    diff = np.abs(gradient_ground_truth[0] - gradient_denoised[0]) + np.abs(gradient_ground_truth[1] - gradient_denoised[1])
    return np.exp(-alpha * np.mean(diff))

def compute_niqe(image):
    # NIQE metric implementation placeholder
    return np.mean(image)  # Placeholder, replace with the actual NIQE computation

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--low_light_dir', default='../input/lol-dataset/lol_dataset/eval15/low', type=str, help='directory of low-light images')
    parser.add_argument('--high_light_dir', default='../input/lol-dataset/lol_dataset/eval15/high', type=str, help='directory of high-light images')
    parser.add_argument('--set_names', default=['normal'], type=list, help='name of test dataset')
    parser.add_argument('--sigma', default=25, type=int, help='noise level')
    parser.add_argument('--model_dir', default='./models/DIAA_sigma25/', type=str, help='directory of the model')
    parser.add_argument('--model_name', default='model_001.keras', type=str, help='the model name')
    parser.add_argument('--result_dir', default='./results', type=str, help='directory of results')
    parser.add_argument('--save_result', default=1, type=int, help='save the enhanced images, 1 or 0')
    return parser.parse_args(args = [])


def load_image_pair(low_light_path, high_light_path):
    low_light_img = np.array(imread(low_light_path), dtype=np.float32) / 255.0
    high_light_img = np.array(imread(high_light_path), dtype=np.float32) / 255.0

    # Resize images if needed (256x256 for the model)
    low_light_img = cv2.resize(low_light_img, (256, 256))
    high_light_img = cv2.resize(high_light_img, (256, 256))
    
    # Ensure the images have 3 channels (RGB)
    if low_light_img.shape[-1] != 3:
        low_light_img = cv2.cvtColor(low_light_img, cv2.COLOR_GRAY2RGB)
    if high_light_img.shape[-1] != 3:
        high_light_img = cv2.cvtColor(high_light_img, cv2.COLOR_GRAY2RGB)  
    return low_light_img, high_light_img

def save_and_display_images(low_light_img, enhanced_img, result_dir, image_name):
    # Scale the float images to 0-255 and convert to uint8
    low_light_img_8bit = (low_light_img * 255).astype(np.uint8)
    enhanced_img_8bit = (enhanced_img * 255).astype(np.uint8)
    
    combined_img = np.hstack((low_light_img_8bit, enhanced_img_8bit))
    
    # Save the image
    imsave(os.path.join(result_dir, image_name + '_comparison.png'), combined_img)
    
   # Display the image using matplotlib
    plt.figure(figsize=(10, 5))
    plt.imshow(combined_img)
    plt.title('Low-Light vs Enhanced')
    plt.axis('off')  # Hide the axis
    plt.show()

args = parse_args()

# =============================================================================
# serialize model to JSON
model_json = model.to_json()
# with open("model.json", "w") as json_file:
with open(os.path.join(args.model_dir,'model.json'), "w") as json_file:
    json_file.write(model_json)
# serialize weights to HDF5
# model.save_weights("model.h5")
model.save_weights(os.path.join(args.model_dir,'.weights.h5'))

print("Saved model")

# =============================================================================

if not os.path.exists(args.result_dir):
    os.mkdir(args.result_dir)

json_file = open(os.path.join(args.model_dir, 'model.json'), 'r')
loaded_model_json = json_file.read()
json_file.close()

keras.config.enable_unsafe_deserialization()

model = model_from_json(loaded_model_json)
model.load_weights(os.path.join(args.model_dir, args.model_name))

psnrs, ssims, cnrs, enls, epis, rmses, foms, niqes = [], [], [], [], [], [], [], []

for im in os.listdir(args.low_light_dir):
    if im.endswith(".jpg") or im.endswith(".png") or im.endswith(".bmp"):
        low_light_img_path = os.path.join(args.low_light_dir, im)
        high_light_img_path = os.path.join(args.high_light_dir, im)  # Assuming the same image names

        low_light_img, high_light_img = load_image_pair(low_light_img_path, high_light_img_path)

        low_light_tensor = np.expand_dims(np.expand_dims(low_light_img, axis=0), axis=-1)

        start_time = time.time()
        enhanced_img = model.predict(low_light_tensor)[0]
        elapsed_time = time.time() - start_time

        print(high_light_img.shape)
        print(enhanced_img.shape)
        
        psnr_x_ = 10 * np.log10(1 / np.mean((high_light_img - enhanced_img) ** 2))
        ssim_x_ = compare_ssim(high_light_img, enhanced_img,data_range=1, channel_axis=-1)
        cnr_x_ = contrast_to_noise_ratio(enhanced_img, high_light_img)
        enl_x_ = equivalent_number_of_looks(enhanced_img)
        epi_x_ = edge_preservation_index(enhanced_img, high_light_img)
        rmse_x_ = root_mean_square_error(enhanced_img, high_light_img)
        fom_x_ = figure_of_merit(enhanced_img, high_light_img)
        niqe_x_ = compute_niqe(enhanced_img)

        psnrs.append(psnr_x_)
        ssims.append(ssim_x_)
        cnrs.append(cnr_x_)
        enls.append(enl_x_)
        epis.append(epi_x_)
        rmses.append(rmse_x_)
        foms.append(fom_x_)
        niqes.append(niqe_x_)

        if args.save_result:
            save_and_display_images(low_light_img, enhanced_img, args.result_dir, os.path.splitext(im)[0])

psnr_avg = np.mean(psnrs)
ssim_avg = np.mean(ssims)
cnr_avg = np.mean(cnrs)
enl_avg = np.mean(enls)
epi_avg = np.mean(epis)
rmse_avg = np.mean(rmses)
fom_avg = np.mean(foms)
niqe_avg = np.mean(niqes)

print(f"PSNR: {psnr_avg}, SSIM: {ssim_avg}, CNR: {cnr_avg}, ENL: {enl_avg}, EPI: {epi_avg}, RMSE: {rmse_avg}, FOM: {fom_avg}, NIQE: {niqe_avg}")

metrics_df = pd.DataFrame({
    'PSNR': [psnr_avg], 'SSIM': [ssim_avg], 'CNR': [cnr_avg], 'ENL': [enl_avg],
    'EPI': [epi_avg], 'RMSE': [rmse_avg], 'FOM': [fom_avg], 'NIQE': [niqe_avg]
})
metrics_df.to_csv(os.path.join(args.result_dir, 'metrics_summary.csv'), index=False)

In [ ]:
metrics_df